## Testing Phase

### Testing on unseen data from original dataset

In [18]:
import sys
sys.path.append('../src')
from feature_engineering import build_features
import pandas as pd
import joblib

# load saved model + all the values you need
saved = joblib.load('../models/isolation_forest_v1.joblib')
model = saved['model']
threshold = saved['threshold']

# load unseen data
df_new = pd.read_csv(r"../data/raw/web_server_access_logs.csv", skiprows=(1,100001), nrows=150000)

# build features — using the SAVED threshold, not a freshly calculated one
df_clean_new = build_features(df_new, size_threshold=saved['size_threshold'])

# prepare X in the exact same column order as training
X_new = df_clean_new.drop(columns=['label', 'type'])[saved['feature_columns']]

# score and predict
scores_new = -model.decision_function(X_new)
predictions = (scores_new >= threshold).astype(int)

# evaluate
y_new_security = df_clean_new['type'].isin(['rce', 'scanning']).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_new_security, predictions))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00    149992
           1       0.00      0.25      0.00         8

    accuracy                           0.99    150000
   macro avg       0.50      0.62      0.50    150000
weighted avg       1.00      0.99      1.00    150000



### Change Threshold

In [22]:
# Deliberately lowered for throttling context — recall prioritized over precision
threshold_final = threshold * 0.85
threshold_final

np.float64(0.0856796511637141)

### Evaluate with Final Threshold

In [23]:
# score and predict
scores_new = -model.decision_function(X_new)
predictions = (scores_new >= threshold_final).astype(int)

# evaluate
y_new_security = df_clean_new['type'].isin(['rce', 'scanning']).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_new_security, predictions))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00    149992
           1       0.01      1.00      0.01         8

    accuracy                           0.99    150000
   macro avg       0.50      1.00      0.50    150000
weighted avg       1.00      0.99      1.00    150000



### Update Saved Model

In [25]:
saved['threshold'] = threshold_final
joblib.dump(saved, '../models/isolation_forest_v1.joblib')

print("Saved Model Updated")

Saved Model Updated
